# SPS Self-Specialization — Colab Test & Capability Inspector

This notebook runs the Stage 1 deterministic type-specialization prototype without Ollama or any AI model.

It verifies tests, runs the specialization demo, manually executes registered capabilities, and lets you inspect the capability registry, hierarchy, source code, events, and persisted storage.

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip install -q -r requirements.txt pytest
!git rev-parse HEAD

## 1. Run the full test suite

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 2. Run the end-to-end self-specialization demo

This starts from `IntegerMultiplication [S0]`, requests float multiplication, and verifies/reuses `FloatMultiplication [S1]`.

In [ ]:
%cd /content/self-specialization
!rm -rf data/capability-registry
!SPS_DEMO_RESET=1 PYTHONPATH=. python experiments/self_specialization_demo.py

## 3. Check the specialization boundary

In [ ]:
from specialization.type_specialization_rules import TYPE_SPECIALIZATION_RULES, is_type_specialization_allowed

print('Declared Stage 1 rules:')
for rule in TYPE_SPECIALIZATION_RULES:
    print(f'  {rule.source_types} -> {rule.target_types} for {rule.operation}')

assert is_type_specialization_allowed('multiply', ['int', 'int'], 'int', 'multiply', ['float', 'float'], 'float')
assert is_type_specialization_allowed('multiply', ['int', 'int'], 'int', 'multiply', ['long', 'long'], 'long')
assert is_type_specialization_allowed('multiply', ['int', 'int'], 'int', 'multiply', ['double', 'double'], 'double')
assert not is_type_specialization_allowed('multiply', ['int', 'int'], 'int', 'add', ['int', 'int'], 'int')
print('\nBoundary check passed:')
print('  int multiplication -> float multiplication: ALLOWED')
print('  multiplication -> addition: REJECTED')

## 4. Manually execute capabilities

`CapabilityRegistry` uses `get(name)` to retrieve a capability. The earlier `get_by_name()` example was incorrect for this implementation.

In [ ]:
from specialization import CapabilityRegistry

registry = CapabilityRegistry.persistent()

integer_mul = registry.get('IntegerMultiplication')
float_mul = registry.get('FloatMultiplication')

print('Integer:', integer_mul.execute(6, 7))
print('Float:', float_mul.execute(2.5, 4.0))

## 5. List all registered capabilities

In [ ]:
from specialization import CapabilityRegistry

registry = CapabilityRegistry.persistent()

print(f'Total capabilities: {len(registry.all())}')
print()
for cap in sorted(registry.all(), key=lambda c: (c.created_at, c.id)):
    print(f'Name:       {cap.name}')
    print(f'ID:         {cap.id}')
    print(f'Version:    {cap.version}')
    print(f'State:      {cap.state}')
    print(f'Input:      {cap.input_types}')
    print(f'Output:     {cap.output_type}')
    print(f'Parent ID:  {cap.parent_id}')
    print(f'Children:   {cap.children_ids}')
    print('-' * 60)

## 6. Show active capabilities

In [ ]:
print('Active capabilities (S1):')
for cap in registry.list_active():
    print(f'  {cap.name} [{cap.state}]')

## 7. Inspect a capability in detail

In [ ]:
import json

details = registry.inspect('FloatMultiplication')
print(json.dumps(details, indent=2))

## 8. Show capability lineage / hierarchy

In [ ]:
float_cap = registry.get('FloatMultiplication')
print('Lineage:')
for cap in registry.lineage(float_cap.id):
    print(f'  {cap.name} [{cap.state}]')

print('\nChildren of SerializeCapability:')
for cap in registry.children(registry.get('SerializeCapability').id):
    print(f'  {cap.name} [{cap.state}]')

## 9. Show generated source code

In [ ]:
print(float_cap.source_code)

## 10. Verify persistence by creating a new registry instance

In [ ]:
reloaded = CapabilityRegistry.persistent()
reloaded_float = reloaded.get('FloatMultiplication')
print('Reloaded:', reloaded_float.name, f'[{reloaded_float.state}]')
print('Reloaded result:', reloaded_float.execute(3.0, 5.0))
print('Registry files:', reloaded_float.inspect_handler()['resources']) if 'resources' in reloaded_float.inspect_handler() else None